# Python Functions — Clean Reference Notebook

**Contents**
1. User-defined functions
2. Returning values
3. Arguments: positional, keyword, default, `*args`, `**kwargs`
4. Scope (local vs. global)
5. Recursion
6. Lambda, `filter()`, `map()`, `reduce()`
7. Errors and exception handling
8. Sets

> Cells that *intentionally* raise errors are wrapped in `try/except`, so the whole notebook runs top to bottom.

## 1. User-defined functions
A function is defined with `def`, can take parameters, and is run by calling it.

In [1]:
def greet():
    """Print a two-line greeting."""
    print("Hello")
    print("World")


greet()

Hello
World


In [2]:
def print_person(name, title):
    """Print a person's name and title."""
    print(f"Name: {name}")
    print(f"Title: {title}")


# One reusable function replaces three near-identical ones.
print_person("Abhishek", "Sahoo")
print_person("Sager", "Mahapatra")
print_person("Ram", "Tiwari")

Name: Abhishek
Title: Sahoo
Name: Sager
Title: Mahapatra
Name: Ram
Title: Tiwari


In [3]:
# Functions are objects too.
print(type(greet))
print(type(print_person))

<class 'function'>
<class 'function'>


## 2. Returning values
`return` sends a result back to the caller. Returning several values actually returns a **tuple**, which you can unpack.

In [4]:
def add(x, y, z=0):
    """Return the sum of the given numbers."""
    return x + y + z


print(add(2, 3))
print(add(2, 3, 3))

5
8


In [5]:
def arithmetic(x, y):
    """Return (sum, difference, product, quotient) of x and y."""
    return x + y, x - y, x * y, x / y


results = arithmetic(4, 5)
print(type(results), results)

total, difference, product, quotient = arithmetic(4, 5)   # tuple unpacking
print(total, difference, product, quotient)

<class 'tuple'> (9, -1, 20, 0.8)
9 -1 20 0.8


## 3. Arguments

| Kind | Meaning |
|---|---|
| Positional | Matched **by position** |
| Keyword | Matched **by parameter name** |
| Default | Parameter has a fallback value |
| Variable-length | `*args` → tuple, `**kwargs` → dict |

### Positional vs. keyword

In [6]:
def student(first_name, last_name):
    print(first_name, last_name)


student("Greeks", "Practice")                         # positional: order matters
student(last_name="Greeks", first_name="Practice")    # keyword: order doesn't matter

Greeks Practice
Practice Greeks


### Arguments are passed by object reference — rebinding a parameter does not change the caller's variable

In [7]:
def add_ten(a):
    a = a + 10
    print("inside the function, a =", a)


x = 10
print("x before call:", x)
add_ten(x)
print("x after call: ", x)   # unchanged

x before call: 10
inside the function, a = 20
x after call:  10


### Common mistakes (shown safely)

In [8]:
def person(name, age, village, district):
    print(name, age * 3, village, district)


# Missing argument -> TypeError
try:
    person("Pinku", 22, "hbd")
except TypeError as e:
    print("TypeError:", e)

# Wrong order -> TypeError (str * int is fine, but str - int is not)
def age_minus_three(name, age):
    print(name)
    print(age - 3)

try:
    age_minus_three(20, "abhi")
except TypeError as e:
    print("TypeError:", e)

TypeError: person() missing 1 required positional argument: 'district'
20
TypeError: unsupported operand type(s) for -: 'str' and 'int'


In [9]:
# A non-default parameter cannot follow a default one -> SyntaxError at compile time.
try:
    compile("def person(name=12, age): pass", "<demo>", "exec")
except SyntaxError as e:
    print("SyntaxError:", e.msg)

SyntaxError: parameter without a default follows parameter with a default


### Default arguments

In [10]:
def person(name="nit", age=20):
    print(name, age)


person()              # both defaults
person("Ravi")        # age default
person("Ravi", 30)    # no defaults
person(age=40)        # name default, age by keyword

nit 20
Ravi 20
Ravi 30
nit 40


### `*args` — any number of positional arguments (a tuple)

In [11]:
def total(first, *rest):
    """Sum `first` and every extra positional argument."""
    return first + sum(rest)


print(total(5, 6, 7, 8))      # 26
print(total(10))              # 10 — rest is empty

26
10


In [12]:
def show_args(first, *rest):
    print("first:", first)
    print("rest :", rest, type(rest))


show_args(10, 20, 30, 40, 50)

first: 10
rest : (20, 30, 40, 50) <class 'tuple'>


### `**kwargs` — any number of keyword arguments (a dict)

In [13]:
def profile(name, **details):
    print(name)
    for key, value in details.items():
        print(f"  {key}: {value}")


profile("Alex", age=36, home_place="southcity", mob=9090)

Alex
  age: 36
  home_place: southcity
  mob: 9090


In [14]:
# `name` is already a named parameter, so it can't also appear inside **kwargs.
def profile_demo(name, **details):
    print(name, details)

try:
    profile_demo("Abhishek", name="Abhishek", village="kamalasing")
except TypeError as e:
    print("TypeError:", e)

profile_demo(name="Abhishek", village="kamalasing", dist="puri")

TypeError: profile_demo() got multiple values for argument 'name'
Abhishek {'village': 'kamalasing', 'dist': 'puri'}


### Taking input — keep `input()` out of the logic so functions stay testable

In [15]:
def make_user(name, age):
    """Build a user record. Raises ValueError for a non-numeric age."""
    return {"name": name, "age": int(age)}


# In an interactive session you would do:
#   user = make_user(input("Name: "), input("Age: "))
print(make_user("Asha", "29"))

{'name': 'Asha', 'age': 29}


## 4. Scope
Variables created inside a function are **local**. Reading a global is fine; *assigning* to one requires `global`.

In [16]:
a = 10

def show_scope():
    b = 20                       # local
    print("inside  -> a:", a, "| b:", b)

show_scope()
print("outside -> a:", a)

try:
    print(b)                     # b only exists inside the function
except NameError as e:
    print("NameError:", e)

inside  -> a: 10 | b: 20
outside -> a: 10
NameError: name 'b' is not defined


In [17]:
counter = 0

def increment():
    global counter               # needed because we assign to it
    counter += 1
    return counter


increment()
increment()
print("counter:", counter)

counter: 2


## 5. Examples: counting, Fibonacci, factorial

In [18]:
def count_even_odd(numbers):
    """Return (even_count, odd_count) for an iterable of ints."""
    even = sum(1 for n in numbers if n % 2 == 0)
    return even, len(numbers) - even


even, odd = count_even_odd([1, 2, 3, 4, 5, 6, 7, 8, 9])
print("even:", even, "| odd:", odd)

even: 4 | odd: 5


In [19]:
def fibonacci(n):
    """Return the first n Fibonacci numbers as a list."""
    sequence = []
    a, b = 0, 1
    for _ in range(n):
        sequence.append(a)
        a, b = b, a + b
    return sequence


print(fibonacci(10))   # [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


In [20]:
def factorial(n):
    """Return n! (iterative)."""
    if n < 0:
        raise ValueError("factorial is not defined for negative numbers")
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result


print(factorial(5))   # 120

120


## 6. Recursion
A function that calls itself needs a **base case**. Without one, Python stops it with `RecursionError`.

In [21]:
import sys

def factorial_recursive(n):
    if n <= 1:                   # base case
        return 1
    return n * factorial_recursive(n - 1)


print(factorial_recursive(5))
print("Recursion limit:", sys.getrecursionlimit())

120
Recursion limit: 3000


In [22]:
def endless(depth=0):
    """No base case -> eventually raises RecursionError."""
    return endless(depth + 1)


old_limit = sys.getrecursionlimit()
sys.setrecursionlimit(150)       # setrecursionlimit (not getrecursionlimit) changes it
try:
    endless()
except RecursionError as e:
    print("RecursionError:", e)
finally:
    sys.setrecursionlimit(old_limit)

RecursionError: maximum recursion depth exceeded


## 7. Lambda, `filter()`, `map()`, `reduce()`

- `lambda args: expression` — a small anonymous function
- `filter(func, items)` — keep items where `func(item)` is truthy
- `map(func, items)` — transform every item
- `functools.reduce(func, items)` — fold items into a single value

In [23]:
nums = [1, 2, 3, 4, 5, 6, 7, 8, 9]

def is_even(n):
    return n % 2 == 0

print(list(filter(is_even, nums)))                  # named function
print(list(filter(lambda n: n % 2 == 0, nums)))     # same thing, lambda
print(list(filter(lambda n: n % 2 != 0, nums)))     # odds
print(list(filter(lambda n: n % 3 == 0, nums)))     # multiples of 3

[2, 4, 6, 8]
[2, 4, 6, 8]
[1, 3, 5, 7, 9]
[3, 6, 9]


In [24]:
evens = list(filter(lambda n: n % 2 == 0, nums))

doubled = list(map(lambda n: n * 2, evens))         # transform
is_odd_flags = list(map(lambda n: n % 2 != 0, nums))  # map returns the *result*, here booleans

print("evens   :", evens)
print("doubled :", doubled)
print("odd flag:", is_odd_flags)

evens   : [2, 4, 6, 8]
doubled : [4, 8, 12, 16]
odd flag: [True, False, True, False, True, False, True, False, True]


In [25]:
from functools import reduce

print(reduce(lambda acc, n: acc + n, nums))   # 45 (sum)
print(reduce(lambda acc, n: acc * n, nums))   # 362880 (product)

45
362880


In [26]:
# filter() keeps items where the function is truthy — it does NOT transform them.
def plus_two(n):
    return n + 2

print(list(filter(plus_two, [-2, 0, 1])))  # [0, 1]: -2 + 2 == 0 is falsy, so -2 is dropped; values are NOT changed
print(list(map(plus_two, [-2, 0, 1])))     # [0, 2, 3]: what you probably wanted

[0, 1]
[0, 2, 3]


In [27]:
# Idiomatic alternative: list comprehensions
print([n for n in nums if n % 2 == 0])
print([n * 2 for n in nums if n % 2 == 0])

[2, 4, 6, 8]
[4, 8, 12, 16]


## 8. Errors

| Type | When | Example |
|---|---|---|
| Syntax (compile-time) | Invalid code, caught before running | `SyntaxError` |
| Runtime | Fails while running | `ZeroDivisionError`, `NameError`, `TypeError` |
| Logical | Runs fine but gives the wrong result | Using `-` instead of `+` |

In [28]:
# Runtime errors
for snippet in ["10 / 0", "undefined_name", "'a' + 1", "int('abc')"]:
    try:
        eval(snippet)
    except Exception as e:
        print(f"{snippet!r:15} -> {type(e).__name__}: {e}")

'10 / 0'        -> ZeroDivisionError: division by zero
'undefined_name' -> NameError: name 'undefined_name' is not defined
"'a' + 1"       -> TypeError: can only concatenate str (not "int") to str
"int('abc')"    -> ValueError: invalid literal for int() with base 10: 'abc'


In [29]:
# Logical error: no exception, wrong answer
def average(a, b):
    return a + b / 2        # BUG: operator precedence

def average_fixed(a, b):
    return (a + b) / 2

print(average(4, 6), "vs", average_fixed(4, 6))

7.0 vs 5.0


### `try` / `except` / `else` / `finally`
- `try` — code that might fail
- `except SomeError as e` — handle a specific error; `e` holds the error
- `else` — runs only if no error occurred
- `finally` — always runs

In [30]:
def safe_divide(a, b):
    """Return a / b, or None if b is zero."""
    try:
        result = a / b
    except ZeroDivisionError as e:
        print(f"Cannot divide {a} by zero ({e})")
        return None
    else:
        print(f"{a} / {b} = {result}")
        return result
    finally:
        print("done\n")


safe_divide(10, 92)
safe_divide(10, 0)

10 / 92 = 0.10869565217391304
done

Cannot divide 10 by zero (division by zero)
done



In [31]:
def parse_non_negative(text):
    """Convert text to a non-negative int, with clear errors."""
    try:
        value = int(text)
    except ValueError:
        raise ValueError(f"{text!r} is not a whole number") from None
    if value < 0:
        raise ValueError("value must be >= 0")
    return value


for sample in ["5", "-3", "abc"]:
    try:
        print(sample, "->", parse_non_negative(sample))
    except ValueError as e:
        print(sample, "->", "ValueError:", e)

5 -> 5
-3 -> ValueError: value must be >= 0
abc -> ValueError: 'abc' is not a whole number


## 9. Sets

In [32]:
s = {1, 2, 3}
s.add(4)              # add() modifies in place and returns None
print(s)

s.discard(99)         # discard() is safe if the item is missing
s.remove(2)           # remove() raises KeyError if missing
print(s)

try:
    s.remove(99)
except KeyError as e:
    print("KeyError:", e)

empty_dict = {}       # note: {} is a dict, not a set
empty_set = set()
print(type(empty_dict), type(empty_set))

{1, 2, 3, 4}
{1, 3, 4}
KeyError: 99
<class 'dict'> <class 'set'>
